# 미션 4-2 | 감성 분석 리소스 탐색 노트북

> **목표:** 설치한 자연어 처리 도구와 데이터를 **직접 불러와 관찰**하고, 이후 감성 분석기 구현에 사용할 입력·출력 형식을 이해한다.

- **다루는 것**
  - 한국어 감성 사전 **KNU SentiLex**: 구조, 감성 점수, 단어 조회
  - 한국어 영화 리뷰 **NSMC**: 열 구성, 레이블 분포, 문장 예시
  - 영어 리뷰 **NLTK Movie Reviews**: 카테고리와 원문
  - **KoNLPy Okt**: 한국어 형태소 분석·품사 태깅
  - **NLTK**: 영어 토큰화·표제어 추출
  - **NLTK VADER**: 영어 감성 사전 및 기존 감성 분석기 호출
- **아직 다루지 않는 것**
  - 감성 사전 정규화·매칭 엔진 구현
  - 부정어/강조어 규칙의 직접 구현
  - 모델 학습 및 정확도·F1 평가

**폴더 전제**

```text
4-2/
├── codyssey_4_2_nlp_environment.py
├── .venv/      # Python 가상환경 및 패키지
├── data/
│   ├── lexicons/ko/SentiWord_info.json
│   ├── datasets/nsmc/ratings_train.txt
│   ├── datasets/nsmc/ratings_test.txt
│   └── nltk_data/
└── notebooks/
    └── 01_nlp_resources_exploration.ipynb  ← 이 파일
```

이 노트북은 **다운로드 또는 설치를 수행하지 않는다.** 설치한 내용을 확인하고 탐색하는 용도다.

## 0. 시작 전에 — 커널과 환경

- 설치 스크립트를 정상 실행했다면 Python 패키지는 루트의 `.venv/`에 있다.
- Jupyter 또는 VS Code에서 노트북을 열어 **해당 가상환경의 Python을 커널로 선택**한다.
- `ipykernel`이 없어 커널로 선택되지 않는 경우, 프로젝트 루트에서 다음을 실행한다.

```bash
source data/activate.sh
python -m pip install ipykernel
```

- **별도의 `sudo`는 필요 없다.** 위 패키지는 가상환경 안에 설치된다.
- 여기서는 노트북 자체에서 `pip install`을 호출하지 않는다. 실행 환경이 달라지는 혼란을 막기 위해서다.
- Java와 NLTK 데이터 경로는 아래 첫 코드 셀에서 **노트북 실행 프로세스에 한해서** 설정한다.

## 1. 경로와 환경변수 준비

- **루트 기준:** `4-2/`
- **모든 데이터 기준:** `4-2/data/`
- 노트북을 `4-2/` 또는 `4-2/notebooks/`에서 열더라도 같은 경로를 참조하도록 설정한다.
- `NLTK_DATA`는 NLTK가 다운로드한 사전을 찾는 데 사용한다.
- `JAVA_HOME`은 KoNLPy의 JVM 초기화에 필요할 수 있다. 로컬 JDK가 있으면 자동으로 탐색하고, 없으면 기존 시스템 Java를 사용한다.

In [ ]:
from pathlib import Path
import os
import platform
import sys
import json

# Jupyter를 시작한 위치(cwd)에서 프로젝트 루트를 찾는다.
# 가정: 'data/'와 'notebooks/'가 같은 루트 바로 아래에 존재한다.
CANDIDATES = (Path.cwd(), *Path.cwd().parents)
ROOT = next(
    (path for path in CANDIDATES if (path / "data").is_dir() and (path / "notebooks").is_dir()),
    None,
)
if ROOT is None:
    raise FileNotFoundError(
        "프로젝트 루트를 찾지 못했습니다. 4-2/ 또는 4-2/notebooks/에서 노트북을 실행하세요."
    )
ROOT = ROOT.resolve()
DATA = ROOT / "data"
NLTK_DIR = DATA / "nltk_data"
KNU_PATH = DATA / "lexicons" / "ko" / "SentiWord_info.json"
NSMC_DIR = DATA / "datasets" / "nsmc"

# NLTK가 검색할 데이터 경로를 고정한다. (시스템 전역 설정은 변경하지 않음)
os.environ["NLTK_DATA"] = str(NLTK_DIR)

# 가상환경 활성화 없이 노트북을 열어도 로컬 JDK를 인식하도록 보완한다.
# 환경변수에 이미 JAVA_HOME이 있다면 우선 존중한다.
manifest_path = DATA / "setup_manifest.json"
manifest = json.loads(manifest_path.read_text(encoding="utf-8")) if manifest_path.is_file() else {}
java_candidates = [DATA / "jdk-25" / "Contents" / "Home", DATA / "jdk-25"]
if manifest.get("java_home"):
    java_candidates.append(Path(manifest["java_home"]))
if not os.environ.get("JAVA_HOME"):
    java_home = next((p for p in java_candidates if (p / "bin" / "java").is_file()), None)
    if java_home is not None:
        os.environ["JAVA_HOME"] = str(java_home)
        os.environ["PATH"] = str(java_home / "bin") + os.pathsep + os.environ.get("PATH", "")

print("프로젝트 루트  :", ROOT)
print("데이터 디렉터리:", DATA)
print("노트북 Python  :", sys.executable)
print("운영체제       :", platform.system(), platform.machine())
print("JAVA_HOME      :", os.environ.get("JAVA_HOME", "(미지정, 시스템 Java 검색)"))
print("NLTK_DATA      :", os.environ["NLTK_DATA"])

# sys.executable의 resolve()는 가상환경 안팎에서 동일한 기본 인터프리터를 가리킬 수 있다.
# 따라서 실제 가상환경 루트(sys.prefix)로 비교한다.
expected_venv = ROOT / ".venv"
if Path(sys.prefix).resolve() != expected_venv.resolve():
    print(f"⚠ 커널의 가상환경: {sys.prefix}")
    print(f"   예상 가상환경: {expected_venv}")
    print("   VS Code에서 4-2/.venv/bin/python 커널을 선택하세요.")

## 2. 설치된 패키지와 파일 확인

- `KoNLPy`, `NLTK`는 **라이브러리**이므로 Python이 임포트할 수 있어야 한다.
- KNU와 NSMC는 **데이터 파일**이므로 경로가 존재해야 한다.
- `NLTK`가 사용하는 리소스는 **Python 패키지와 별개**로 `data/nltk_data/`에 있어야 한다.
- 아래 셀은 상태를 보고하는 것이 목적이며, 없는 파일을 자동 다운로드하지 않는다.

In [ ]:
from importlib.metadata import PackageNotFoundError, version

missing_packages = []
for package in ("konlpy", "JPype1", "nltk"):
    try:
        print(f"{package:10s}: {version(package)}")
    except PackageNotFoundError:
        missing_packages.append(package)
        print(f"{package:10s}: 미설치 (커널/설치 대상 경로 확인)")

files_to_check = {
    "KNU 한국어 감성사전": KNU_PATH,
    "NSMC 학습 데이터": NSMC_DIR / "ratings_train.txt",
    "NSMC 테스트 데이터": NSMC_DIR / "ratings_test.txt",
    "NLTK 데이터 디렉터리": NLTK_DIR,
}
for name, path in files_to_check.items():
    status = "OK" if path.exists() else "없음"
    print(f"{name:20s}: {status}  ({path.relative_to(ROOT)})")

if missing_packages:
    print("\n⚠ 현재 노트북 커널에는 필요한 Python 패키지가 없습니다.")
    print("  프로젝트 루트에서 수정된 설치 스크립트를 실행하세요:")
    print("  .venv/bin/python codyssey_4_2_nlp_environment.py")
    if (DATA / ".venv").is_dir():
        print("  ※ data/.venv도 발견되었습니다. 이전 설치 스크립트가 여기에 패키지를 설치했을 수 있습니다.")

if manifest:
    print("설치 매니페스트: 있음 (플랫폼:", manifest.get("platform", "?"), ")")
else:
    print("설치 매니페스트: 없음 (설치 자체가 실패했다는 뜻은 아님)")

## 3. KNU 한국어 감성 사전 탐색

**감성 사전의 역할**

- **`word`**: 사전에 등록된 표면 단어 또는 표현
- **`word_root`**: 데이터가 제공하는 어근 관련 값
- **`polarity`**: 감성 점수 (일반적으로 음수=부정, 양수=긍정, 0=중립)

**살펴볼 것**

- 전체 항목 수는 얼마나 되는가?
- 점수별로 항목이 얼마나 분포하는가?
- 단일 단어만 있는가, 여러 단어로 된 표현도 있는가?

> **주의:** 이 사전의 `word_root`를 Kiwi 또는 Okt의 출력과 무조건 같은 표제어로 간주하면 안 된다. 우선 원본 값을 살펴본 후 매칭 규칙을 정한다.

In [ ]:
from collections import Counter

if not KNU_PATH.is_file():
    raise FileNotFoundError(f"KNU 감성 사전이 없습니다: {KNU_PATH}\n설치 스크립트 실행 후 다시 확인하세요.")

# .json이므로 pandas.read_csv가 아니라 json.load를 사용한다.
with KNU_PATH.open("r", encoding="utf-8-sig") as source:
    knu_rows = json.load(source)

print("자료형:", type(knu_rows).__name__)
print("전체 항목 수:", len(knu_rows))
print("처음 5개 항목:")
for row in knu_rows[:5]:
    print(" -", row)

polarity_counts = Counter(str(row.get("polarity", "")) for row in knu_rows)
print("\n점수별 항목 수:")
for score, count in sorted(polarity_counts.items(), key=lambda p: int(p[0])):
    print(f" - {score:>2}: {count:,}개")

In [ ]:
# 사전에서 단어를 직접 찾을 때 편리하도록 word -> 점수 색인을 만들어 본다.
# 동일 word가 여러 번 나올 가능성을 확인하기 위해 중복 여부도 따로 조사한다.
knu_lookup = {}
duplicate_words = set()
for row in knu_rows:
    word = row["word"]
    if word in knu_lookup:
        duplicate_words.add(word)
    knu_lookup[word] = int(row["polarity"])

print("서로 다른 표면 표현 수:", len(knu_lookup))
print("중복된 표면 표현 수:", len(duplicate_words))
for word in ("좋다", "나쁘다", "실망", "최고", "정말"):
    # None은 사전에 해당 표현이 없다는 의미. 0점과 구별한다.
    print(f" - {word!r}: {knu_lookup.get(word)}")

# 아직 복합 표현을 토큰 단위로 분해하지 않는다.
phrase_examples = [row["word"] for row in knu_rows if " " in row["word"]][:5]
print("공백을 포함한 표현 예시:", phrase_examples)

## 4. NSMC 영화 리뷰 데이터 탐색

- `ratings_train.txt`, `ratings_test.txt`는 **탭(`\t`)으로 구분된 TSV** 파일이다.
- 각 행은 `id`, `document`, `label`로 구성된다.
  - `label=1`: 긍정
  - `label=0`: 부정
- 파일이 크므로 **모든 문장을 리스트로 저장하지 않고 순차적으로 읽는다.**
- 원본은 학습/테스트가 이미 구분되어 있다. 비교 실험에서도 테스트 데이터는 개발 중 규칙 조정에 사용하지 않는 것이 좋다.

In [ ]:
import csv

def inspect_nsmc(path: Path, examples_per_label: int = 2):
    """TSV를 한 줄씩 읽어 건수/레이블 분포/짧은 예시만 반환한다."""
    counts = Counter()
    samples = {"0": [], "1": []}
    empty_text_count = 0

    if not path.is_file():
        raise FileNotFoundError(f"NSMC 파일이 없습니다: {path}")
    with path.open("r", encoding="utf-8", newline="") as source:
        reader = csv.DictReader(source, delimiter="\t")
        print(f" - {path.name} 열 이름:", reader.fieldnames)
        for row in reader:
            label = row["label"]
            counts[label] += 1
            text = (row["document"] or "").strip()
            if not text:
                empty_text_count += 1
            if label in samples and text and len(samples[label]) < examples_per_label:
                samples[label].append(text)

    return counts, samples, empty_text_count

for file_name in ("ratings_train.txt", "ratings_test.txt"):
    print(f"\n[{file_name}]")
    counts, samples, empty_count = inspect_nsmc(NSMC_DIR / file_name)
    print(" - 레이블별 건수:", dict(counts))
    print(" - 총 문장 수:", sum(counts.values()))
    print(" - 빈 텍스트 수:", empty_count)
    for label, descriptions in samples.items():
        print(f" - {'긍정' if label == '1' else '부정'} 리뷰 예시:")
        for sentence in descriptions:
            print("   •", sentence[:120])

## 5. NLTK가 다운로드한 데이터 연결하기

- `NLTK`는 실행할 Python 패키지이고, `vader_lexicon`, `movie_reviews`, `wordnet`, `punkt_tab`은 별도로 받는 **데이터 리소스**다.
- 설정 스크립트는 위 리소스를 모두 `data/nltk_data/`에 저장하도록 설계되어 있다.
- 이제 Python에서 NLTK의 리소스 검색 경로를 확인하고, 각 리소스의 존재를 검사한다.

In [ ]:
import nltk

# 노트북 커널 시작 전에 NLTK_DATA를 설정하지 않았어도 해당 위치를 탐색하게 한다.
if str(NLTK_DIR) not in nltk.data.path:
    nltk.data.path.insert(0, str(NLTK_DIR))

resources = {
    "VADER 감성 사전": "sentiment/vader_lexicon.zip",
    "영어 영화 리뷰": "corpora/movie_reviews",
    "영어 표제어 사전": "corpora/wordnet.zip",
    "토큰화 리소스": "tokenizers/punkt_tab",
}
for label, resource_name in resources.items():
    try:
        found = nltk.data.find(resource_name)
        print(f" - {label}: OK ({found})")
    except LookupError:
        print(f" - {label}: 미발견 ({resource_name})")

print("NLTK 검색 경로 (앞의 3개):", nltk.data.path[:3])

## 6. NLTK Movie Reviews — 영어 평가 데이터

- `movie_reviews` 코퍼스에는 긍정(`pos`)과 부정(`neg`) 리뷰 파일이 있다.
- NSMC처럼 하나의 TSV가 아니라 **개별 파일들의 집합**이다.
- 이후 평가 시 NSMC의 0/1 레이블과 `neg`/`pos` 레이블을 동일한 기준으로 변환하면 비교가 가능하다.

In [ ]:
from nltk.corpus import movie_reviews

fileids = movie_reviews.fileids()
print("전체 문서 수:", len(fileids))
for category in movie_reviews.categories():
    category_ids = movie_reviews.fileids(category)
    print(f" - {category}: {len(category_ids):,}건")
    sample_id = category_ids[0]
    preview = " ".join(movie_reviews.raw(sample_id).split())[:260]
    print(f"   파일 ID: {sample_id}")
    print(f"   원문 일부: {preview}...")

## 7. KoNLPy Okt — 한국어 형태소 분석

- 입력 문장을 **형태소와 품사**로 나누어 반환한다.
- `stem=False`는 표면 형태를 더 가깝게 보여주고, `stem=True`는 일부 활용형을 원형에 가깝게 변환한다.
- **관찰 포인트:** 감성 사전의 `word`와 분석기의 결과가 그대로 일치하는가?
- 이 출력은 형태소 분석 결과이지, **아직 감성 분석 결과가 아니다.**

In [ ]:
from konlpy.tag import Okt

okt = Okt()
korean_sentences = [
    "나는 영화를 봤다.",
    "영화가 정말 재미있었어요.",
    "이 영화는 좋지 않았어요.",
]
for sentence in korean_sentences:
    print("\n원문:", sentence)
    print(" - 품사 분석:", okt.pos(sentence, norm=True, stem=False))
    print(" - 어간 정규화:", okt.pos(sentence, norm=True, stem=True))

## 8. NLTK — 영어 토큰화와 표제어 추출

- **토큰화:** 문장을 토큰으로 분할한다.
- **표제어 추출(lemmatization):** `movies → movie`처럼 사전의 기본형에 가까운 표기를 얻는다.
- WordNet 표제어 추출은 **품사를 지정했는지**에 따라 결과가 달라질 수 있다.
- 한국어 Okt와 영어 NLTK는 같은 종류의 처리를 수행하더라도 토큰 형태와 품사 체계가 다르다.

In [ ]:
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

english_sentence = "The movies were surprisingly good, but the endings were disappointing."
tokens = word_tokenize(english_sentence)
lemmatizer = WordNetLemmatizer()

print("원문:", english_sentence)
print("토큰:", tokens)
print("명사 기본형(movies):", lemmatizer.lemmatize("movies", pos="n"))
print("동사 기본형(running):", lemmatizer.lemmatize("running", pos="v"))
print("동사라고 지정하지 않은 running:", lemmatizer.lemmatize("running"))

## 9. NLTK VADER — 영어 감성 사전 조회와 완성형 분석기

VADER에는 **서로 구분할 두 기능**이 있다.

- **감성 사전 조회:** `SentimentIntensityAnalyzer().lexicon`에서 단어별 점수를 확인한다.
- **완성형 감성 분석:** `polarity_scores()`는 사전 점수 외에 여러 휴리스틱 규칙을 적용한다.

> 미션 4-2에서는 감성 점수 합산·부정어·강조어 규칙을 **직접 구현**해야 한다. 아래의 `polarity_scores()`는 완성형 도구를 **관찰·비교하기 위한 참고**이지, 과제의 구현을 대신하지 않는다.

In [ ]:
from nltk.sentiment.vader import SentimentIntensityAnalyzer

vader = SentimentIntensityAnalyzer()
print("VADER 사전 항목 수:", len(vader.lexicon))
for word in ("good", "excellent", "bad", "boring", "movie"):
    print(f" - {word:10s}: {vader.lexicon.get(word)}")

vader_samples = [
    "This movie is good.",
    "This movie is VERY good!",
    "This movie is not good.",
    "This movie is bad.",
]
for sentence in vader_samples:
    print("\n문장:", sentence)
    print(" - 감성 결과:", vader.polarity_scores(sentence))

print("\n※ compound는 문장 단위로 정규화된 종합 감성 점수(-1~1)다.")

## 10. 실험: 형태소 분석 결과와 KNU 사전 키를 단순 비교

- 형태소 분석기가 반환한 문자열을 KNU `word`에 그대로 조회해 보자.
- **일치하지 않는다고 곧바로 중립인 것은 아니다.** 활용형·표제어·복합 표현 등의 차이 때문일 수 있다.
- 아래는 **매칭 구조 관찰 실험**이며 최종 감성 분석기는 아니다.
- 특히 부정어의 적용 범위나 강조어의 가중치는 아직 계산하지 않는다.

In [ ]:
example = "영화가 정말 좋지 않았어요."
original_tokens = okt.pos(example, norm=True, stem=False)
stemmed_tokens = okt.pos(example, norm=True, stem=True)

print("입력 문장:", example)
for mode, token_pairs in (("표면형", original_tokens), ("기본형 변환", stemmed_tokens)):
    print(f"\n[{mode}]")
    for token, pos in token_pairs:
        score = knu_lookup.get(token)  # 없으면 None; 0점 항목과 구분
        print(f" - {token:<12} / {pos:<10} / 사전 점수: {score}")

print("\n해석할 때의 주의점:")
print(" - 점수가 없더라도 문장 전체에 감성이 없다고 단정할 수 없습니다.")
print(" - KNU 다중 어절 표현, 부정어, 강조어는 아직 처리하지 않았습니다.")

## 11. 정리와 다음 확장 방향

**이번 노트북에서 확인한 것**

- **KNU:** 감성 표현과 점수가 기록된 원본 데이터
- **NSMC:** 한국어 리뷰 원문과 0/1 정답 레이블
- **NLTK Movie Reviews:** 영어 리뷰 원문과 `pos`/`neg` 레이블
- **Okt:** 한국어 문장을 형태소·품사로 분해
- **NLTK Tokenizer/WordNet:** 영어 토큰화·표제어 추출
- **VADER:** 영어 감성 사전과 완성형 분석기의 결과 확인

**다음에 직접 구현할 부분**

1. 원본 사전(`word`)과 토큰화 결과가 일치하도록 **매칭·정규화 정책** 수립
2. 한 단어뿐 아니라 **여러 형태소로 된 감성 표현** 탐지
3. **부정어 범위**와 **강조어 가중치** 적용
4. 표현별 점수 합산, 최종 레이블 판정
5. NSMC 등 평가 데이터로 **Accuracy / F1 / 실패 사례** 분석

**복습 질문**

- 감성 사전과 평가 데이터셋은 왜 다른가?
- `좋다`가 사전에 있어도 `좋았어요`가 바로 매칭되지 않을 수 있는 이유는?
- VADER를 그대로 호출하는 것과 4-2에서 직접 규칙을 작성하는 것은 어떻게 다른가?

> 이 노트북의 코드 셀은 원본 데이터와 설치 상태를 확인하는 **읽기 중심의 탐색**이므로, 여러 번 실행해도 학습 데이터를 변경하지 않는다.